In [0]:
# ---- imports.py ----
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

print("✅ Context injector imports done")

✅ Context injector imports done


In [0]:
# ---- fetch_scores.py ----
def fetch_opportunity_scores(product_code):
    """
    Fetch top 5 countries for a given HS product code
    from your gold opportunity scores table
    """
    df = spark.sql(f"""
        SELECT
            ReportingCountry,
            continent,
            ProductCode,
            ProductDescription,
            AVE,
            NoOfTariffLines,
            distance,
            exchange_rate_to_inr,
            total_import_value_usd,
            num_trade_partners,
            has_import_data,
            demand_score,
            risk_score,
            opportunity_score,
            market_quadrant,
            reason
        FROM workspace.gold.gold_opportunity_scores
        WHERE CAST(ProductCode AS STRING) = '{product_code}'
        ORDER BY opportunity_score DESC 
        LIMIT 5
    """)

    rows = [row.asDict() for row in df.collect()]
    print(f"✅ Fetched {len(rows)} countries for product {product_code}")
    return rows

# Test
scores = fetch_opportunity_scores("120991")
for s in scores:
    print(f"  {s['ReportingCountry']} | Score: {s['opportunity_score']} | {s['market_quadrant']}")

✅ Fetched 5 countries for product 120991
  Bahrain | Score: 1.7971 | Star Market
  Afghanistan | Score: 1.073 | Star Market
  China | Score: 0.9875 | Star Market
  Azerbaijan | Score: 0.966 | Star Market
  Bangladesh | Score: 0.9617 | Star Market


In [0]:
# ---- build_context.py ----
def build_country_context(scores):
    """
    Convert scores list into readable context string
    to inject into RAG chain
    """
    if not scores:
        return "No opportunity score data available for this product."

    lines = ["=== Export Opportunity Analysis ===\n"]

    for i, row in enumerate(scores):
        import_val = row.get("total_import_value_usd", 0)
        import_m   = round(import_val / 1_000_000, 2) if import_val else 0

        lines.append(f"""
Rank {i + 1}: {row['ReportingCountry']} ({row['continent']})
- Product        : {row.get('ProductDescription', 'N/A')[:80]}
- Market Quadrant: {row.get('market_quadrant', 'Unknown')}
- Opportunity    : {row.get('opportunity_score', 0)} | Demand: {row.get('demand_score', 0)} | Risk: {row.get('risk_score', 0)}
- Market Size    : ${import_m}M imports from {row.get('num_trade_partners', 0)} suppliers
- Tariff (AVE)   : {row.get('AVE', 'N/A')}
- Distance       : {row.get('distance', 'N/A')} km from India
- Exchange Rate  : 1 USD = {row.get('exchange_rate_to_inr', 'N/A')} INR
- Analysis       : {row.get('reason', 'No detailed analysis available.')}
        """)

    return "\n".join(lines)

# Test
context_str = build_country_context(scores)
print(context_str)

=== Export Opportunity Analysis ===


Rank 1: Bahrain (Asia)
- Product        : Seeds, fruit and spores, of a kind used for sowing : Other : Vegetable seeds
- Market Quadrant: Star Market
- Opportunity    : 1.7971 | Demand: 1.0454 | Risk: -1.5035
- Market Size    : $2.05M imports from 62 suppliers
- Tariff (AVE)   : 0.0
- Distance       : 2723.0 km from India
- Exchange Rate  : 1 USD = 220.0 INR
- Analysis       : Market: Emerging ($2.05M imports, 62 suppliers). Demand: Strong — low tariffs and close proximity. Risk: Very Low — strong currency and short distance. 
        

Rank 2: Afghanistan (Asia)
- Product        : Seeds, vegetable, nes for sowing
- Market Quadrant: Star Market
- Opportunity    : 1.073 | Demand: 0.9216 | Risk: -0.3028
- Market Size    : $14.96M imports from 63 suppliers
- Tariff (AVE)   : 0.02500000037252903
- Distance       : 1578.0 km from India
- Exchange Rate  : 1 USD = 1.0 INR
- Analysis       : Market: Medium ($15.0M imports, 63 suppliers). Demand: Strong — l

In [0]:
# ---- inject_context.py ----
def inject_context_into_query(user_question, product_code):
    """
    Enriches user question with opportunity score context
    so RAG chain can answer why questions
    """
    scores      = fetch_opportunity_scores(product_code)
    context_str = build_country_context(scores)

    enriched_query = f"""
{user_question}

Use this export opportunity data to support your answer:
{context_str}
    """
    return enriched_query, scores

# Test
enriched, scores = inject_context_into_query(
    "Why should I export to Bahrain?",
    "120991"
)
print(enriched)

✅ Fetched 5 countries for product 120991

Why should I export to Bahrain?

Use this export opportunity data to support your answer:
=== Export Opportunity Analysis ===


Rank 1: Bahrain (Asia)
- Product        : Seeds, fruit and spores, of a kind used for sowing : Other : Vegetable seeds
- Market Quadrant: Star Market
- Opportunity    : 1.7971 | Demand: 1.0454 | Risk: -1.5035
- Market Size    : $2.05M imports from 62 suppliers
- Tariff (AVE)   : 0.0
- Distance       : 2723.0 km from India
- Exchange Rate  : 1 USD = 220.0 INR
- Analysis       : Market: Emerging ($2.05M imports, 62 suppliers). Demand: Strong — low tariffs and close proximity. Risk: Very Low — strong currency and short distance. 
        

Rank 2: Afghanistan (Asia)
- Product        : Seeds, vegetable, nes for sowing
- Market Quadrant: Star Market
- Opportunity    : 1.073 | Demand: 0.9216 | Risk: -0.3028
- Market Size    : $14.96M imports from 63 suppliers
- Tariff (AVE)   : 0.02500000037252903
- Distance       : 1578.0 k